# Telemetry Data Pipeline Runner

This notebook orchestrates the execution of the telemetry data pipeline.

Each stage corresponds to a notebook implementing part of the Medallion architecture:

Bronze -> Silver -> Gold

The runner executes notebooks sequentially and stops the pipeline if any stage fails.

In [0]:
# 00_pipeline_runner
# Orchestrates the execution of the telemetry data pipeline

import time
from datetime import datetime

# ---------------------------
# Pipeline configuration
# ---------------------------

BUILD_PIPELINE = [
    ("00_setup_and_schema", "Initialize database and Delta tables"),
    ("10_bronze_transit_ingest", "Ingest public transport telemetry events"),
    ("20_silver_transit_metrics", "Compute transit performance metrics"),
    ("30_bronze_weather_ingest", "Ingest FMI weather observations"),
    ("40_silver_weather_metrics", "Aggregate weather metrics"),
    ("50_gold_route_kpi_window", "Generate route-level KPIs by window"),
    ("60_gold_route_kpi_daily", "Generate daily route-level KPIs"),
    ("70_gold_pipeline_metrics_window", "Build pipeline-level operational metrics"),
]

# Fail fast on first error
FAIL_FAST = True

# Dry run prints the execution plan without running notebooks
DRY_RUN = False


# ---------------------------
# Helper functions
# ---------------------------

def now_str() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def print_pipeline_plan(pipeline_steps):
    print("===== Pipeline Execution Plan =====")
    for i, (nb, desc) in enumerate(pipeline_steps, start=1):
        print(f"{i:02d}. {nb}  |  {desc}")
    print("===================================\n")


# ---------------------------
# Execute pipeline
# ---------------------------

pipeline = BUILD_PIPELINE
results = []

pipeline_start_ts = time.time()

print(f"===== Starting telemetry pipeline at {now_str()} =====")
print_pipeline_plan(pipeline)

if DRY_RUN:
    print("DRY_RUN=True -> no notebooks were executed.")
else:
    for step_idx, (nb, desc) in enumerate(pipeline, start=1):
        print(f"\n===== Step {step_idx}/{len(pipeline)}: {nb} =====")
        print(f"Description: {desc}")

        stage_start_ts = time.time()
        stage_start_str = now_str()

        try:
            result = dbutils.notebook.run(nb, timeout_seconds=0)
            stage_end_str = now_str()
            duration_sec = round(time.time() - stage_start_ts, 2)

            print(
                f"===== Finished {nb} | "
                f"Status: SUCCESS | "
                f"Duration: {duration_sec}s | "
                f"Result: {result} ====="
            )

            results.append({
                "step": step_idx,
                "notebook": nb,
                "description": desc,
                "status": "SUCCESS",
                "result": result,
                "error_message": None,
                "started_at": stage_start_str,
                "ended_at": stage_end_str,
                "duration_sec": duration_sec,
            })

        except Exception as e:
            stage_end_str = now_str()
            duration_sec = round(time.time() - stage_start_ts, 2)
            error_message = str(e)

            print(
                f"===== Failed {nb} | "
                f"Status: FAILED | "
                f"Duration: {duration_sec}s | "
                f"Error: {error_message} ====="
            )

            results.append({
                "step": step_idx,
                "notebook": nb,
                "description": desc,
                "status": "FAILED",
                "result": None,
                "error_message": error_message,
                "started_at": stage_start_str,
                "ended_at": stage_end_str,
                "duration_sec": duration_sec,
            })

            if FAIL_FAST:
                print("\nPipeline stopped because FAIL_FAST=True.")
                break


# ---------------------------
# Final summary
# ---------------------------

total_duration_sec = round(time.time() - pipeline_start_ts, 2)

print("\n===== Pipeline Summary =====")
for row in results:
    if row["status"] == "SUCCESS":
        print(
            f"{row['step']:02d}. {row['notebook']} | "
            f"{row['status']} | "
            f"{row['duration_sec']}s | "
            f"{row['result']}"
        )
    else:
        print(
            f"{row['step']:02d}. {row['notebook']} | "
            f"{row['status']} | "
            f"{row['duration_sec']}s | "
            f"{row['error_message']}"
        )

print(f"\n===== Total Pipeline Duration: {total_duration_sec}s =====")

===== Starting telemetry pipeline at 2026-03-11 20:24:39 =====
===== Pipeline Execution Plan =====
01. 00_setup_and_schema  |  Initialize database and Delta tables
02. 10_bronze_transit_ingest  |  Ingest public transport telemetry events
03. 20_silver_transit_metrics  |  Compute transit performance metrics
04. 30_bronze_weather_ingest  |  Ingest FMI weather observations
05. 40_silver_weather_metrics  |  Aggregate weather metrics
06. 50_gold_route_kpi_window  |  Generate route-level KPIs by window
07. 60_gold_route_kpi_daily  |  Generate daily route-level KPIs
08. 70_gold_pipeline_metrics_window  |  Build pipeline-level operational metrics


===== Step 1/8: 00_setup_and_schema =====
Description: Initialize database and Delta tables
===== Finished 00_setup_and_schema | Status: SUCCESS | Duration: 61.46s | Result: OK =====

===== Step 2/8: 10_bronze_transit_ingest =====
Description: Ingest public transport telemetry events
===== Finished 10_bronze_transit_ingest | Status: SUCCESS | Durati